# Manuscript figures 2–4

Builds the three main-text figures as vector PDF + SVG and a 300 dpi PNG preview, dumps the
exact plotted values per panel to `figures/figure_data/`, and asserts that every summary
statistic it computes matches the number quoted in the manuscript.

The assertions run **before** the plotting cell for each figure. A mismatch raises with the
computed and the expected value side by side and nothing is drawn — the figure is wrong or
the manuscript is wrong, and that is not a decision this notebook makes.

| Figure | Panels | Reads |
|---|---|---|
| 2 — assembly strategy comparison | a contigs, b total length | `quality_control_and_routing.tsv` + `../pipeline_tables/assembly_stats.tsv` |
| 3 — mapping rate distribution | single | `quality_control_and_routing.tsv` |
| 4 — *Klebsiella* read fraction | a genus fraction, b top non-*Klebsiella* | `quality_control_and_routing.tsv` + `../pipeline_tables/kraken2_summary.tsv` |

In [1]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator, FixedFormatter, NullLocator

TABLES = Path("../pipeline_tables")   # the stage summary tables this repo ships

SRC = {
    "s5": TABLES / "quality_control_and_routing.tsv",
    "asm_stats": TABLES / "assembly_stats.tsv",
    "kraken2": TABLES / "kraken2_summary.tsv",
}
missing = [f"{k} -> {v}" for k, v in SRC.items() if not Path(v).exists()]
if missing:
    raise SystemExit("missing inputs:\n  " + "\n  ".join(missing))

FIGDIR = Path("figures")
DATADIR = FIGDIR / "figure_data"
FIGDIR.mkdir(parents=True, exist_ok=True)
DATADIR.mkdir(parents=True, exist_ok=True)

SEED = 20260810          # jitter RNG seed; changing it changes the figures
print(f"{len(SRC)} inputs present | jitter seed {SEED}")

3 inputs present | jitter seed 20260810


## Shared styling

Okabe–Ito, colourblind-safe. The mapping is fixed once here and reused across all three
figures so a colour means the same thing throughout.

In [2]:
MM = 1 / 25.4                       # Scientific Data: 89 mm single column, 183 mm double

COLOUR = {
    "ont": "#0072B2",               # Nanopore-only
    "illumina": "#D55E00",          # Illumina-only
    "hybrid": "#009E73",            # hybrid
    "released": "#4D4D4D",          # released libraries / kraken2 PASS
    "excluded": "#D55E00",          # excluded libraries / kraken2 FAIL
    "threshold": "#999999",         # threshold lines
    "annot": "#666666",             # small grey annotation text
}


def apply_style():
    '''Scientific Data house style. Called by every figure.'''
    plt.rcdefaults()
    plt.rcParams.update({
        # Arial is not installed here; Liberation Sans is metric-compatible with it and
        # both output formats keep the text editable, so the typesetter can substitute.
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
        "font.size": 8,
        "axes.labelsize": 8,
        "axes.titlesize": 8,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 7,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.linewidth": 0.8,
        "axes.grid": False,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 2,
        "ytick.major.size": 2,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "lines.linewidth": 0.8,
        "figure.dpi": 300,
        "savefig.dpi": 300,
        "pdf.fonttype": 42,          # embed as TrueType, text stays selectable
        "ps.fonttype": 42,
        "svg.fonttype": "none",      # SVG keeps live <text>, not outlines
    })


def panel_letter(ax, letter, x=-0.16, y=1.06):
    '''9 pt bold lowercase letter, top left, outside the axes.'''
    ax.text(x, y, letter, transform=ax.transAxes, fontsize=9, fontweight="bold",
            ha="left", va="bottom")


def threshold_line(ax, x, label, y, dx=3, ha="left"):
    '''Dashed grey rule behind the data, with a 7 pt caption offset in display space.

    dx is in points, so the caption sits on the same side of the line whichever way the
    axis runs — both strip figures invert their x axis.
    '''
    ax.axvline(x, color=COLOUR["threshold"], linestyle="--", linewidth=0.8, zorder=0)
    ax.annotate(label, xy=(x, y), xycoords=ax.get_xaxis_transform(),
                xytext=(dx, 0), textcoords="offset points",
                fontsize=7, color=COLOUR["annot"], ha=ha, va="center")


def save_figure(fig, stem):
    '''PDF + SVG + 300 dpi PNG preview, all with the same bounding box.'''
    written = []
    for ext in ("pdf", "svg", "png"):
        path = FIGDIR / f"{stem}.{ext}"
        fig.savefig(path, dpi=300, bbox_inches="tight", pad_inches=0.02)
        written.append(f"{path}  ({path.stat().st_size / 1024:.0f} kB)")
    plt.close(fig)
    print("wrote:\n  " + "\n  ".join(written))


def dump_panel(frame, name):
    '''The exact plotted values, one CSV per panel, for checking and for Zenodo.'''
    path = DATADIR / f"{name}.csv"
    frame.to_csv(path, index=False)
    print(f"panel data: {path}  ({len(frame)} rows, {len(frame.columns)} cols)")


apply_style()
_face = font_manager.findfont(font_manager.FontProperties(
    family=plt.rcParams["font.sans-serif"]))
print(f"style applied | sans-serif resolves to {Path(_face).stem}")

style applied | sans-serif resolves to LiberationSans-Regular


## Assertion harness

`report()` prints one line per series in the form the build spec asks for and records any
mismatch; `assert_all()` raises with every mismatch listed, computed against expected. It
is called at the end of each figure's data cell, so a failure stops before anything plots.

In [3]:
EPS = 1e-9      # so a value that rounds exactly onto the tolerance boundary passes


class FigureChecks:
    '''Collects the statistics one figure quotes and compares them to the manuscript.'''

    def __init__(self, figure):
        self.figure = figure
        self.failures = []
        self.n_checks = 0

    def report(self, panel, series, items):
        '''items: (name, computed, expected_or_None, tolerance, format) tuples.'''
        parts, bad = [], []
        for name, got, expected, tol, fmt in items:
            parts.append(f"{name}={fmt.format(got)}")
            if expected is None:
                continue
            self.n_checks += 1
            if abs(float(got) - float(expected)) > float(tol) + EPS:
                bad.append(f"{name}: computed {fmt.format(got)}, "
                           f"manuscript {fmt.format(expected)}")
        print(f"{self.figure} {panel} — {series}: " + " ".join(parts)
              + (" [OK]" if not bad else " [MISMATCH]"))
        if bad:
            self.failures.append((panel, series, bad))

    def assert_all(self):
        if self.failures:
            lines = [f"{self.figure}: {len(self.failures)} series do not match the manuscript"]
            for panel, series, bad in self.failures:
                lines.append(f"  {panel} — {series}")
                lines.extend(f"      {b}" for b in bad)
            lines.append("  Nothing was plotted. Either the figure inputs are wrong or the "
                         "manuscript number is — resolve it by hand, do not widen the tolerance.")
            raise AssertionError("\n".join(lines))
        print(f"{self.figure}: all {self.n_checks} checks match the manuscript\n")


INT = "{:.0f}"
F1 = "{:.1f}"
F2 = "{:.2f}"

## Figure 2 — assembly strategy comparison

Supports: *Illumina-only assemblies are far more fragmented (mean 112.5 contigs, median 95,
maximum 390) while Nanopore-only and hybrid assemblies reduce each genome to ~2.8 contigs,
at comparable total assembly length (means 5.53, 5.45 and 5.53 Mbp).*

The Nanopore-only series comes from the `ont_*` columns of the routing table, **not** from
`03_assembly` directly. Those columns record the *carried-forward* Nanopore assembly — the
one each hybrid actually derived from, including the decontaminated and Autocycler routes.
Reading `../pipeline_tables/assembly_stats.tsv` for the ONT series would silently give the
pre-remediation 2.9 contigs / 5.60 Mbp and contradict the text.

In [4]:
s5 = pd.read_csv(SRC["s5"], sep="\t")

asm = pd.read_csv(SRC["asm_stats"], sep="\t")
asm["library"] = asm.file.str.split("/").str[1]
asm["strategy"] = asm.file.str.split("/").str[2]
illumina = (asm[asm.strategy == "illumina_unicycler"]
            .set_index("library")[["num_seqs", "sum_len"]]
            .rename(columns={"num_seqs": "illumina_contigs", "sum_len": "illumina_total_len"}))

released = s5[s5.released == "yes"].copy()
fig2 = released.merge(illumina, left_on="library", right_index=True,
                      how="left", validate="1:1")

# the join is the one step outside the supplementary tables, so check it explicitly
assert len(fig2) == 294, f"join yielded {len(fig2)} rows, expected 294"
null_cols = {c: int(fig2[c].isna().sum())
             for c in ["ont_contigs", "ont_total_len", "hybrid_contigs", "hybrid_total_len",
                       "illumina_contigs", "illumina_total_len"] if fig2[c].isna().any()}
assert not null_cols, f"nulls after join: {null_cols}"

STRATEGIES = [("Nanopore-only", "ont", "ont_contigs", "ont_total_len"),
              ("Illumina-only", "illumina", "illumina_contigs", "illumina_total_len"),
              ("hybrid", "hybrid", "hybrid_contigs", "hybrid_total_len")]

EXPECT_F2 = {                        # label: (contig mean, median, max, length mean Mbp)
    "Nanopore-only": (2.8, 2, 13, 5.53),
    "Illumina-only": (112.5, 95, 390, 5.45),
    "hybrid":        (2.8, 2, 16, 5.53),
}

c2 = FigureChecks("Fig 2")
for label, key, ccol, lcol in STRATEGIES:
    contigs, length = fig2[ccol].astype(float), fig2[lcol].astype(float) / 1e6
    e_mean, e_med, e_max, e_len = EXPECT_F2[label]
    c2.report("panel a", label, [
        ("n", len(contigs), 294, 0, INT),
        ("mean", contigs.mean(), e_mean, 0.05, F1),
        ("median", contigs.median(), e_med, 0, INT),
        ("max", contigs.max(), e_max, 0, INT),
    ])
    c2.report("panel b", label, [
        ("n", len(length), 294, 0, INT),
        ("mean_Mbp", length.mean(), e_len, 0.005, F2),
        ("min_Mbp", length.min(), None, 0, F2),
        ("max_Mbp", length.max(), None, 0, F2),
    ])
c2.assert_all()

Fig 2 panel a — Nanopore-only: n=294 mean=2.8 median=2 max=13 [OK]
Fig 2 panel b — Nanopore-only: n=294 mean_Mbp=5.53 min_Mbp=5.09 max_Mbp=6.49 [OK]
Fig 2 panel a — Illumina-only: n=294 mean=112.5 median=95 max=390 [OK]
Fig 2 panel b — Illumina-only: n=294 mean_Mbp=5.45 min_Mbp=4.79 max_Mbp=6.10 [OK]
Fig 2 panel a — hybrid: n=294 mean=2.8 median=2 max=16 [OK]
Fig 2 panel b — hybrid: n=294 mean_Mbp=5.53 min_Mbp=5.10 max_Mbp=6.49 [OK]
Fig 2: all 18 checks match the manuscript



In [5]:
apply_style()
rng = np.random.default_rng(SEED)

fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(183 * MM, 75 * MM))
positions = np.arange(len(STRATEGIES))
JITTER = 0.28

panels = [(ax_a, "a", "ont_contigs", "illumina_contigs", "hybrid_contigs",
           "Contigs per assembly (log scale)"),
          (ax_b, "b", "ont_total_len", "illumina_total_len", "hybrid_total_len",
           "Total assembly length (Mbp)")]

dumps = {"a": [], "b": []}

for ax, letter, *_ , ylabel in panels:
    scale = 1.0 if letter == "a" else 1e-6
    for pos, (label, key, ccol, lcol) in zip(positions, STRATEGIES):
        col = ccol if letter == "a" else lcol
        values = fig2[col].astype(float).to_numpy() * scale
        offsets = rng.uniform(-JITTER, JITTER, size=values.size)

        ax.scatter(pos + offsets, values, s=4, alpha=0.35, linewidths=0,
                   color=COLOUR[key], zorder=2)
        ax.boxplot([values], positions=[pos], widths=0.32, showfliers=False,
                   patch_artist=True, zorder=3,
                   boxprops=dict(facecolor="white", edgecolor="black", linewidth=0.8),
                   medianprops=dict(color="black", linewidth=1.1),
                   whiskerprops=dict(color="black", linewidth=0.8),
                   capprops=dict(color="black", linewidth=0.8))

        dumps[letter].append(pd.DataFrame({
            "library": fig2.library.to_numpy(),
            "isolate": fig2.isolate.to_numpy(),
            "strategy": label,
            ("contigs" if letter == "a" else "total_length_mbp"): values,
            "x_position": pos,
            "x_jitter": offsets,
        }))

    ax.set_xticks(positions)
    ax.set_xticklabels([s[0] for s in STRATEGIES])
    ax.set_xlim(-0.6, len(STRATEGIES) - 0.4)
    ax.set_ylabel(ylabel)
    panel_letter(ax, letter, x=-0.18)

# panel a — log y, the range is 1 to 390
ax_a.set_yscale("log")
YTICKS = [1, 2, 5, 10, 20, 50, 100, 200, 400]
ax_a.yaxis.set_major_locator(FixedLocator(YTICKS))
ax_a.yaxis.set_major_formatter(FixedFormatter([str(t) for t in YTICKS]))
ax_a.yaxis.set_minor_locator(NullLocator())
ax_a.set_ylim(0.8, 560)

# panel b — linear, limits from the data. Starting at zero would flatten all three
# distributions into one line and hide the point that the strategies agree.
lengths = np.concatenate([fig2[c].to_numpy() / 1e6
                          for _, _, _, c in STRATEGIES])
ax_b.set_ylim(lengths.min() - 0.12, lengths.max() + 0.12)

fig.subplots_adjust(wspace=0.28)
save_figure(fig, "Fig2_assembly_strategy_comparison")
dump_panel(pd.concat(dumps["a"], ignore_index=True), "Fig2_panel_a_contigs")
dump_panel(pd.concat(dumps["b"], ignore_index=True), "Fig2_panel_b_total_length")

wrote:
  figures/Fig2_assembly_strategy_comparison.pdf  (43 kB)
  figures/Fig2_assembly_strategy_comparison.svg  (212 kB)
  figures/Fig2_assembly_strategy_comparison.png  (167 kB)
panel data: figures/figure_data/Fig2_panel_a_contigs.csv  (882 rows, 6 cols)
panel data: figures/figure_data/Fig2_panel_b_total_length.csv  (882 rows, 6 cols)


## Figure 3 — mapping rate distribution

Supports: *across the 316 screened libraries the median Illumina-to-Nanopore mapping rate
was 99.56% (mean 97.63%); the seven libraries below 90% were all non-retained colony picks
sharing a short-read library with their paired morphology; across the 294 released
libraries the median was 99.62% (mean 99.14%, minimum 93.58%).*

A histogram is the wrong form here: the mass sits between 99% and 100% and the interesting
structure is seven points below 90%. Instead, one point per library on a single axis, with
the rate on a **logit** scale — the 99–100% mass spreads out and the low outliers stay on
the same axis, with no axis break to explain.

Logit rather than the log of `100 − mapped_pct`: a log-complement axis spreads the upper
tail but crushes the lower one, and the seven libraries between 11.7% and 16.9% land within
0.6% of the axis width of each other, one indistinguishable blob hard against the left
limit. Logit is the natural scale for a proportion, spreads both tails, and happens to put
the tick marks at even spacing. Swapping back is one line — `set_xscale`, the tick
positions, and the axis limits.

In [6]:
fig3 = s5[["library", "isolate", "mapped_pct", "released", "kraken2_status",
           "internal_qc_decision"]].copy()
assert fig3.mapped_pct.notna().all(), "null mapped_pct"
assert fig3.mapped_pct.max() < 100, "a mapping rate of exactly 100% has no log complement"

rel3 = fig3[fig3.released == "yes"].mapped_pct
below90 = fig3[fig3.mapped_pct < 90]

c3 = FigureChecks("Fig 3")
c3.report("panel a", "all screened libraries", [
    ("n", len(fig3), 316, 0, INT),
    ("median", fig3.mapped_pct.median(), 99.56, 0.005, F2),
    ("mean", fig3.mapped_pct.mean(), 97.63, 0.005, F2),
])
c3.report("panel a", "released libraries", [
    ("n", len(rel3), 294, 0, INT),
    ("median", rel3.median(), 99.62, 0.005, F2),
    ("mean", rel3.mean(), 99.14, 0.005, F2),
    ("min", rel3.min(), 93.58, 0.005, F2),
])
c3.report("panel a", "excluded libraries", [
    ("n", (fig3.released == "no").sum(), 22, 0, INT),
])
# the two that carry the corrected claim in the text
c3.report("panel a", "below 90%", [
    ("n", len(below90), 7, 0, INT),
    ("n_released", (below90.released == "yes").sum(), 0, 0, INT),
])
c3.report("panel a", "below 95%", [
    ("n", (fig3.mapped_pct < 95).sum(), 12, 0, INT),
])
c3.assert_all()

Fig 3 panel a — all screened libraries: n=316 median=99.56 mean=97.63 [OK]
Fig 3 panel a — released libraries: n=294 median=99.62 mean=99.14 min=93.58 [OK]
Fig 3 panel a — excluded libraries: n=22 [OK]
Fig 3 panel a — below 90%: n=7 n_released=0 [OK]
Fig 3 panel a — below 95%: n=12 [OK]
Fig 3: all 11 checks match the manuscript



In [7]:
apply_style()
rng = np.random.default_rng(SEED)

fig, ax = plt.subplots(figsize=(89 * MM, 65 * MM))

fig3["mapped_fraction"] = fig3.mapped_pct / 100      # the plotted x, logit needs (0, 1)
fig3["outcome"] = np.where(fig3.released == "yes", "released", "excluded")
fig3["y_jitter"] = rng.uniform(0.05, 0.95, size=len(fig3))

for outcome, colour in [("released", COLOUR["released"]), ("excluded", COLOUR["excluded"])]:
    sub = fig3[fig3.outcome == outcome]
    ax.scatter(sub.mapped_fraction, sub.y_jitter, s=5, alpha=0.6, linewidths=0,
               color=colour, zorder=2, label=outcome)

ax.set_xscale("logit")
# 95% is dropped from the spec's tick list — its label collides with 90% on a single
# column, and the 95% rule is named by its own caption anyway
RATE_TICKS = [10, 50, 90, 99, 99.9]
ax.xaxis.set_major_locator(FixedLocator([r / 100 for r in RATE_TICKS]))
ax.xaxis.set_major_formatter(FixedFormatter([f"{r:g}" for r in RATE_TICKS]))
ax.xaxis.set_minor_locator(NullLocator())
ax.set_xlim(0.05, 0.99995)            # clear of the 11.69% minimum and the 99.99% maximum
ax.set_xlabel("Illumina-to-Nanopore mapping rate (%)")

# captions sit in the free band above the cloud and run away from each other, so the two
# rules do not fight over the same space
# the rules are named in full because only one of them carries an axis tick
threshold_line(ax, 0.95, "95% internal QC threshold", y=1.14, dx=3, ha="left")
threshold_line(ax, 0.90, "n = 7 below 90%", y=1.14, dx=-3, ha="right")

ax.set_ylim(-0.05, 1.26)
ax.yaxis.set_visible(False)
ax.spines["left"].set_visible(False)

ax.legend(loc="upper left", frameon=False, fontsize=7, handletextpad=0.3,
          borderaxespad=0.0, labelspacing=0.3, markerscale=1.8)

save_figure(fig, "Fig3_mapping_rate_distribution")
dump_panel(fig3[["library", "isolate", "mapped_pct", "mapped_fraction", "released",
                 "outcome", "internal_qc_decision", "y_jitter"]],
           "Fig3_panel_a_mapping_rate")

wrote:
  figures/Fig3_mapping_rate_distribution.pdf  (17 kB)
  figures/Fig3_mapping_rate_distribution.svg  (42 kB)
  figures/Fig3_mapping_rate_distribution.png  (60 kB)


panel data: figures/figure_data/Fig3_panel_a_mapping_rate.csv  (316 rows, 8 cols)


## Figure 4 — *Klebsiella* read fraction and contamination flags

Supports: *300 of the 316 libraries (94.9%) passed contamination screening without a flag;
libraries were flagged as carrying non-*Klebsiella* reads (a non-*Klebsiella* genus ≥ 5% of
classified reads), as mixed *Klebsiella* cultures (two species each ≥ 20%), or as low-yield
(*Klebsiella* fraction < 50%).*

The routing table carries the two fractions but not which rule fired, so the flag strings
come from `../pipeline_tables/kraken2_summary.tsv` — the same screen, keyed by
library. The two thresholds need two axes: 50% of the *Klebsiella* fraction in panel a,
5% of the highest non-*Klebsiella* genus in panel b. Panel a is on a logit scale, as in
Fig. 3; panel b is a plain log, because its mass is at the bottom of the range rather than
the top and there is no upper tail to rescue. The mixed-*Klebsiella* rule is a
property of a pair of species and does not plot on either axis, so those libraries are
marked with an open triangle in panel a and the criterion is stated in the legend.

In [8]:
kraken = pd.read_csv(SRC["kraken2"], sep="\t")
assert len(kraken) == 316, f"kraken2 summary has {len(kraken)} rows, expected 316"

fig4 = s5[["library", "isolate", "klebsiella_genus_read_fraction", "top_non_klebsiella_genus",
           "top_non_klebsiella_genus_fraction", "kraken2_status", "released"]].copy()
fig4 = fig4.merge(kraken[["sample", "status", "flags"]].rename(columns={"sample": "library"}),
                  on="library", how="left", validate="1:1")
assert fig4.status.notna().all(), "kraken2 summary does not cover every library"
assert (fig4.status == fig4.kraken2_status).all(), "kraken2 status disagrees with the routing table"

# Are the fractions proportions or percentages? Assert rather than assume.
for col in ["klebsiella_genus_read_fraction", "top_non_klebsiella_genus_fraction"]:
    assert fig4[col].notna().all(), f"null {col}"
    assert fig4[col].max() <= 1.0, f"{col} max is {fig4[col].max()}, not a proportion"
    assert fig4[col].min() > 0, f"{col} has a non-positive value, no log complement"
LOW_YIELD, NON_KLEB = 0.50, 0.05

flags = fig4["flags"].fillna("")
fig4["low_yield"] = flags.str.contains("LOW_KLEBSIELLA")
fig4["non_klebsiella"] = flags.str.contains("NON_KLEBSIELLA_CONTAMINATION")
fig4["mixed_klebsiella"] = flags.str.contains("MIXED_KLEBSIELLA")

c4 = FigureChecks("Fig 4")
c4.report("both panels", "screened libraries", [
    ("n", len(fig4), 316, 0, INT),
    ("n_PASS", (fig4.kraken2_status == "PASS").sum(), 300, 0, INT),
    ("pct_PASS", 100 * (fig4.kraken2_status == "PASS").mean(), 94.9, 0.05, F1),
    ("n_flagged", (fig4.kraken2_status != "PASS").sum(), 16, 0, INT),
])
# each threshold must reproduce the count of the flag it stands for
c4.report("panel a", "low-yield (Klebsiella fraction < 50%)", [
    ("n_below_threshold", (fig4.klebsiella_genus_read_fraction < LOW_YIELD).sum(),
     int(fig4.low_yield.sum()), 0, INT),
    ("n_flagged_LOW_KLEBSIELLA", fig4.low_yield.sum(), 6, 0, INT),
])
c4.report("panel a", "mixed Klebsiella (two species each >= 20%)", [
    ("n_flagged_MIXED_KLEBSIELLA", fig4.mixed_klebsiella.sum(), 7, 0, INT),
])
c4.report("panel b", "non-Klebsiella genus >= 5%", [
    ("n_above_threshold", (fig4.top_non_klebsiella_genus_fraction >= NON_KLEB).sum(),
     int(fig4.non_klebsiella.sum()), 0, INT),
    ("n_flagged_NON_KLEBSIELLA", fig4.non_klebsiella.sum(), 9, 0, INT),
])
c4.report("both panels", "flag union covers every failure", [
    ("n_flag_union", (fig4.low_yield | fig4.non_klebsiella | fig4.mixed_klebsiella).sum(),
     (fig4.kraken2_status != "PASS").sum(), 0, INT),
])
c4.assert_all()

Fig 4 both panels — screened libraries: n=316 n_PASS=300 pct_PASS=94.9 n_flagged=16 [OK]
Fig 4 panel a — low-yield (Klebsiella fraction < 50%): n_below_threshold=6 n_flagged_LOW_KLEBSIELLA=6 [OK]
Fig 4 panel a — mixed Klebsiella (two species each >= 20%): n_flagged_MIXED_KLEBSIELLA=7 [OK]
Fig 4 panel b — non-Klebsiella genus >= 5%: n_above_threshold=9 n_flagged_NON_KLEBSIELLA=9 [OK]
Fig 4 both panels — flag union covers every failure: n_flag_union=16 [OK]
Fig 4: all 10 checks match the manuscript



In [9]:
apply_style()
rng = np.random.default_rng(SEED)

fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(183 * MM, 65 * MM))

fig4["top_non_kleb_pct"] = 100 * fig4.top_non_klebsiella_genus_fraction
fig4["y_jitter_a"] = rng.uniform(0.05, 0.95, size=len(fig4))
fig4["y_jitter_b"] = rng.uniform(0.05, 0.95, size=len(fig4))

STATUS = [("PASS", COLOUR["released"]), ("FAIL", COLOUR["excluded"])]

# --- panel a: Klebsiella genus read fraction, logit ---------------------------------
# Logit rather than the log of the complement, for the same reason as Fig. 3 and more
# sharply: six libraries sit between 0.7% and 29.5%, and a log-complement axis stacks
# them into one blob against the left limit — exactly the libraries the panel is about.
for status, colour in STATUS:
    sub = fig4[(fig4.kraken2_status == status) & ~fig4.mixed_klebsiella]
    ax_a.scatter(sub.klebsiella_genus_read_fraction, sub.y_jitter_a, s=5, alpha=0.6,
                 linewidths=0, color=colour, zorder=2)
mixed = fig4[fig4.mixed_klebsiella]
ax_a.scatter(mixed.klebsiella_genus_read_fraction, mixed.y_jitter_a, s=13, marker="^",
             facecolors="none", edgecolors=COLOUR["excluded"], linewidths=0.7,
             alpha=0.9, zorder=3)

ax_a.set_xscale("logit")
FRAC_TICKS = [1, 10, 50, 90, 99]
ax_a.xaxis.set_major_locator(FixedLocator([t / 100 for t in FRAC_TICKS]))
ax_a.xaxis.set_major_formatter(FixedFormatter([f"{t:g}" for t in FRAC_TICKS]))
ax_a.xaxis.set_minor_locator(NullLocator())
ax_a.set_xlim(0.004, 0.996)           # clear of the 0.66% minimum and the 99.31% maximum
ax_a.set_xlabel("$\\it{Klebsiella}$ genus read fraction (%)")
threshold_line(ax_a, LOW_YIELD, "low-yield threshold", y=1.00, dx=3, ha="left")

n_pass = int((fig4.kraken2_status == "PASS").sum())
n_fail = int((fig4.kraken2_status != "PASS").sum())
n_mixed = int(fig4.mixed_klebsiella.sum())
handles_a = [
    Line2D([], [], marker="o", linestyle="", markersize=3.2, alpha=0.6,
           color=COLOUR["released"], label=f"PASS (n = {n_pass})"),
    Line2D([], [], marker="o", linestyle="", markersize=3.2, alpha=0.6,
           color=COLOUR["excluded"], label=f"flagged (n = {n_fail})"),
    Line2D([], [], marker="^", linestyle="", markersize=4.0, markerfacecolor="none",
           markeredgecolor=COLOUR["excluded"], markeredgewidth=0.7,
           label=f"mixed $\\it{{Klebsiella}}$ (n = {n_mixed})"),
]
ax_a.legend(handles=handles_a, loc="upper left", frameon=False, fontsize=7,
            handletextpad=0.3, borderaxespad=0.0, labelspacing=0.3)

# --- panel b: highest non-Klebsiella genus fraction, direct log ---------------------
for status, colour in STATUS:
    sub = fig4[fig4.kraken2_status == status]
    ax_b.scatter(sub.top_non_kleb_pct, sub.y_jitter_b, s=5, alpha=0.6, linewidths=0,
                 color=colour, zorder=2)

ax_b.set_xscale("log")
B_TICKS = [0.1, 1, 5, 10, 50, 100]
ax_b.xaxis.set_major_locator(FixedLocator(B_TICKS))
ax_b.xaxis.set_major_formatter(FixedFormatter([f"{t:g}" for t in B_TICKS]))
ax_b.xaxis.set_minor_locator(NullLocator())
ax_b.set_xlim(0.04, 190)
ax_b.set_xlabel("Highest non-$\\it{Klebsiella}$ genus fraction (%)")
threshold_line(ax_b, 100 * NON_KLEB, "non-$\\it{Klebsiella}$ threshold",
               y=1.24, dx=-3, ha="right")

# Label the genera above the threshold, not the individual points: nine libraries are
# above 5% but they are only two genera, so nine identical labels would be noise. Each
# genus is labelled once and led back to a representative library — the one nearest the
# genus's geometric mean, which keeps the leaders apart, since the highest library of
# each genus would put both leaders within a millimetre of each other. Labels stack in
# the free band above the cloud, where the text has room to run left.
above = fig4[fig4.top_non_klebsiella_genus_fraction >= NON_KLEB]


def representative(group):
    centre = np.exp(np.log(group.top_non_kleb_pct).mean())
    return group.iloc[(group.top_non_kleb_pct - centre).abs().argmin()]


labelled = (above.groupby("top_non_klebsiella_genus", group_keys=False)
                 .apply(representative)
                 .sort_values("top_non_kleb_pct"))
assert len(labelled) <= 6, f"{len(labelled)} labels above the threshold, at most 6 allowed"

# groupby moves the key to the index and drops the column, so the genus is read off
# the iteration key rather than the row.
for k, (genus, row) in enumerate(labelled.iterrows()):
    n_genus = int((above.top_non_klebsiella_genus == genus).sum())
    ax_b.annotate(f"$\\it{{{genus}}}$ (n = {n_genus})",
                  xy=(row.top_non_kleb_pct, row.y_jitter_b),
                  xytext=(row.top_non_kleb_pct, 1.12 - 0.12 * k), textcoords="data",
                  fontsize=6.5, color=COLOUR["annot"], ha="right", va="center",
                  arrowprops=dict(arrowstyle="-", linewidth=0.5,
                                  color=COLOUR["threshold"], shrinkA=1, shrinkB=2))

for ax, letter in [(ax_a, "a"), (ax_b, "b")]:
    ax.set_ylim(-0.05, 1.30)
    ax.yaxis.set_visible(False)
    ax.spines["left"].set_visible(False)
    panel_letter(ax, letter, x=-0.06)

fig.subplots_adjust(wspace=0.16)
save_figure(fig, "Fig4_klebsiella_read_fraction")
dump_panel(fig4[["library", "isolate", "klebsiella_genus_read_fraction", "kraken2_status",
                 "low_yield", "mixed_klebsiella", "released", "y_jitter_a"]],
           "Fig4_panel_a_klebsiella_fraction")
dump_panel(fig4[["library", "isolate", "top_non_klebsiella_genus",
                 "top_non_klebsiella_genus_fraction", "top_non_kleb_pct", "non_klebsiella",
                 "kraken2_status", "released", "y_jitter_b"]],
           "Fig4_panel_b_top_non_klebsiella")

wrote:
  figures/Fig4_klebsiella_read_fraction.pdf  (36 kB)
  figures/Fig4_klebsiella_read_fraction.svg  (100 kB)
  figures/Fig4_klebsiella_read_fraction.png  (152 kB)
panel data: figures/figure_data/Fig4_panel_a_klebsiella_fraction.csv  (316 rows, 8 cols)
panel data: figures/figure_data/Fig4_panel_b_top_non_klebsiella.csv  (316 rows, 9 cols)


## Outputs

In [10]:
for path in sorted(FIGDIR.glob("Fig*")) + sorted(DATADIR.glob("*.csv")):
    print(f"{path.stat().st_size / 1024:8.0f} kB  {path}")

      43 kB  figures/Fig2_assembly_strategy_comparison.pdf
     167 kB  figures/Fig2_assembly_strategy_comparison.png
     212 kB  figures/Fig2_assembly_strategy_comparison.svg
      17 kB  figures/Fig3_mapping_rate_distribution.pdf
      60 kB  figures/Fig3_mapping_rate_distribution.png
      42 kB  figures/Fig3_mapping_rate_distribution.svg
      36 kB  figures/Fig4_klebsiella_read_fraction.pdf
     152 kB  figures/Fig4_klebsiella_read_fraction.png
     100 kB  figures/Fig4_klebsiella_read_fraction.svg
      46 kB  figures/figure_data/Fig2_panel_a_contigs.csv
      52 kB  figures/figure_data/Fig2_panel_b_total_length.csv
      22 kB  figures/figure_data/Fig3_panel_a_mapping_rate.csv
      19 kB  figures/figure_data/Fig4_panel_a_klebsiella_fraction.csv
      23 kB  figures/figure_data/Fig4_panel_b_top_non_klebsiella.csv
